In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random

# Create sliders for width, height, and humans
width_slider = widgets.IntSlider(value=5, min=1, max=25, description='Width:')
height_slider = widgets.IntSlider(value=5, min=1, max=25, description='Height:')
humans_slider = widgets.IntSlider(value=0, min=0, max=10, description='Humans:') # New slider for humans


# Create a clear button
clear_button = widgets.Button(description="Clear Grid")
# Create a generate home button
generate_home_button = widgets.Button(description="Generate Home")


# Display sliders, clear button and initial grid
display(width_slider)
display(height_slider)
display(humans_slider) # Display the new humans slider
display(clear_button)
display(generate_home_button) # Display the new generate home button


color_counts = {'info': 0, 'success': 0, 'primary': 0, 'warning': 0, 'danger': 0, '': 0}
color_labels = {'info': 'Blue', 'success': 'Green', 'primary': 'Dark Blue', 'warning': 'Yellow', 'danger': 'Red', '': 'Default'}
color_count_labels = {}
# Removed adjacent_colored_cubes_label


def update_color_counts():
    """Updates the color counts based on the current grid state."""
    global color_counts
    global color_count_labels
    color_counts = {'info': 0, 'success': 0, 'primary': 0, 'warning': 0, 'danger': 0, '': 0}
    for row in grid_widget.children:
        for button in row.children:
            color_counts[button.button_style] += 1
    for color, count in color_counts.items():
        color_count_labels[color].value = f"{color_labels[color]}: {count}"
    check_human_requirements({'new': humans_slider.value}) # Call the new function to check requirements with the current value


def get_connected_colored_cubes():
    """Finds connected colored cubes and returns a dictionary of counts for the largest connected component of colored cubes."""
    rows = len(grid_widget.children)
    cols = len(grid_widget.children[0].children) if rows > 0 else 0
    visited = set()
    connected_counts = {'info': 0, 'success': 0, 'primary': 0, 'warning': 0, 'danger': 0}
    largest_component_counts = {'info': 0, 'success': 0, 'primary': 0, 'warning': 0, 'danger': 0}

    def dfs(r, c, current_component_counts):
        if (r, c) in visited or r < 0 or r >= rows or c < 0 or c >= cols:
            return
        button = grid_widget.children[r].children[c]
        color = button.button_style
        if color == '':
            return

        visited.add((r, c))
        if color in current_component_counts:
          current_component_counts[color] += 1

        for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            dfs(r + dr, c + dc, current_component_counts)

    for r in range(rows):
        for c in range(cols):
            if (r, c) not in visited:
                current_component_counts = {'info': 0, 'success': 0, 'primary': 0, 'warning': 0, 'danger': 0}
                dfs(r, c, current_component_counts)
                # Compare current component counts with largest component counts
                if sum(current_component_counts.values()) > sum(largest_component_counts.values()):
                  largest_component_counts = current_component_counts


    return largest_component_counts



def check_human_requirements(change):
    """Checks if there are enough connected cubes for the number of humans."""
    required_cubes_per_human = {
        'info': 2,      # Blue: thermal control and life support
        'success': 1,   # Green: medical care and Communications
        'warning': 1,   # Yellow: sleep and exercise
        'primary': 1,   # Dark Blue: waste management and power
        'danger': 1     # Red: stowage, food storage, and preparation
    }
    humans = change['new']
    message = ""
    if humans > 0:
        connected_counts = get_connected_colored_cubes()
        message += f"With {humans} humans, you need to increase the following cubes in the main system:\n"
        needs_increase = False
        for color, required_per_human in required_cubes_per_human.items():
            required_total = humans * required_per_human
            current_connected_count = connected_counts.get(color, 0)
            if current_connected_count < required_total:
                message += f"- {color_labels[color]}: Need {required_total - current_connected_count} more\n"
                needs_increase = True
        if not needs_increase:
            message = f"With {humans} humans, you have enough of all required cube types in the main system."
    else:
        message = "" # Clear message if no humans

    human_requirements_label.value = message


def generate_home(b):
    """Generates a home layout based on human requirements."""
    clear_grid(None)  # Clear the current grid

    required_cubes_per_human = {
        'info': 2,
        'success': 1,
        'warning': 1,
        'primary': 1,
        'danger': 1
    }
    humans = humans_slider.value
    total_required_cubes = {color: humans * required_per_human for color, required_per_human in required_cubes_per_human.items()}

    rows = height_slider.value
    cols = width_slider.value
    placed_cubes = []
    colors = ['info', 'success', 'primary', 'warning', 'danger', ''] # Define colors here

    # Place colored cubes adjacently
    current_row, current_col = 0, 0
    for color, count in total_required_cubes.items():
        placed_count = 0
        while placed_count < count:
            if current_row < rows and current_col < cols:
                button = grid_widget.children[current_row].children[current_col]
                button.button_style = color
                button.color_index = colors.index(color)
                placed_cubes.append((current_row, current_col))
                placed_count += 1
                current_col += 1
                if current_col >= cols:
                    current_col = 0
                    current_row += 1
            else:
                # If grid is full before placing all required cubes, stop
                break

    update_color_counts()


# Create labels for color counts
for color in color_counts:
    color_count_labels[color] = widgets.Label(value=f"{color_labels[color]}: 0")
    display(color_count_labels[color])

# Label for human requirements message
human_requirements_label = widgets.Label(value="")
display(human_requirements_label)
# Removed display of adjacent_colored_cubes_label


def on_square_click(b):
    """Changes the button color when clicked."""
    colors = ['info', 'success', 'primary', 'warning', 'danger', '']
    current_color_index = b.color_index
    next_color_index = (current_color_index + 1) % len(colors)
    b.button_style = colors[next_color_index]
    b.color_index = next_color_index
    update_color_counts()

def create_square_button(row, col):
    """Creates a button representing a square."""
    button = widgets.Button(
        description='',
        button_style='', # Initial color is default
        layout=widgets.Layout(width='50px', height='50px', margin='1px')
    )
    button.color_index = 5 # Initialize color index to point to the default color
    button.on_click(on_square_click)
    return button

def create_grid(rows, cols):
    """Creates a grid of square buttons."""
    grid = []
    for row in range(rows):
        row_buttons = []
        for col in range(cols):
            row_buttons.append(create_square_button(row, col))
        grid.append(widgets.HBox(row_buttons))
    return widgets.VBox(grid)

def update_grid(change):
    """Clears the output and displays the updated grid."""
    global grid_widget
    with output_area:
        clear_output(wait=True)
        grid_widget = create_grid(height_slider.value, width_slider.value)
        display(grid_widget)
        update_color_counts()


def clear_grid(b):
    """Resets all grid squares to the default color."""
    for row in grid_widget.children:
        for button in row.children:
            button.button_style = ''
            button.color_index = 5 # Reset color index to point to the default color
    update_color_counts()


# Observe the sliders and update the grid to reflect the changes
width_slider.observe(update_grid, names='value')
height_slider.observe(update_grid, names='value')
humans_slider.observe(check_human_requirements, names='value') # Observe the new humans slider


# Initial grid display
output_area = widgets.Output()
display(output_area)
grid_widget = create_grid(height_slider.value, width_slider.value)
with output_area:
    clear_output(wait=True) # Clear previous grid before displaying the new one
    display(grid_widget)
update_color_counts() # Update counts and check requirements initially


clear_button.on_click(clear_grid)
generate_home_button.on_click(generate_home) # Observe the new generate home button

IntSlider(value=5, description='Width:', max=25, min=1)

IntSlider(value=5, description='Height:', max=25, min=1)

IntSlider(value=0, description='Humans:', max=10)

Button(description='Clear Grid', style=ButtonStyle())

Button(description='Generate Home', style=ButtonStyle())

...

Label(value='Yellow: 0')

Label(value='Red: 0')

Label(value='Default: 0')

Label(value='')

Output()